# KalariSena — GEM-X retargeting on Colab GPUConverts Kalari human videos into Unitree G1 motions.**What this notebook does**```your videos  ->  GEM-X (GPU)  ->  *_retarget_g1.csv  ->  NPZ  ->  download```Then on your Mac you run the two scripts that are already built and tested:`render_g1_motion.py` (robot videos) and `build_review_page.py` (side-by-side page).**Honesty note, please read.** I have not been able to run GEM-X — it needs CUDA andwas not installed on the machine this notebook was written on. The kalariSena parts(clone, batch loop, annotate, packaging) follow your repo's own code exactly andshould be correct. The **GEM-X install and checkpoint cells are the uncertain ones**:step 3 prints GEM-X's own install docs so you can adapt rather than trust my guess.Expect to iterate on cells 3-5. Everything after the CSVs exist is solid.**First: Runtime -> Change runtime type -> GPU (T4 is fine).**

## 1. Confirm the GPU

In [ ]:
!nvidia-smiimport torch, sysprint("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())if not torch.cuda.is_available():    print("\n*** NO GPU. Runtime -> Change runtime type -> T4 GPU, then rerun. ***")print("python:", sys.version.split()[0])

## 2. Clone kalariSena + GEM-XGEM-X uses submodules, so `--recursive` matters.

In [ ]:
import osos.chdir("/content")KALARI_REPO = ""   # optional: your kalariSena git URL. Leave "" to upload scripts manually.if KALARI_REPO and not os.path.isdir("/content/kalariSena"):    !git clone $KALARI_REPO /content/kalariSenaelif not os.path.isdir("/content/kalariSena"):    os.makedirs("/content/kalariSena/scripts", exist_ok=True)    print("No repo URL set. Upload annotate_motion_library.py into "          "/content/kalariSena/scripts/ and the G1 URDF+meshes into "          "/content/kalariSena/assets/unitree_g1/ (use the Files pane on the left).")if not os.path.isdir("/content/GEM-X"):    !git clone --recursive https://github.com/NVlabs/GEM-X.git /content/GEM-Xelse:    print("GEM-X already cloned")!ls /content/GEM-X

## 3. Read GEM-X's own install instructions**Do not skip this.** I could not verify GEM-X's setup, so read what it actuallysays here and adapt cell 4 accordingly. Its requirements may conflict with Colab'spreinstalled torch.

In [ ]:
import os, globfor p in ["/content/GEM-X/README.md",          "/content/GEM-X/docs/INSTALL.md",          "/content/GEM-X/docs/INSTALL_MACOS.md"]:    if os.path.exists(p):        print("=" * 78); print(p); print("=" * 78)        print(open(p, errors="replace").read()[:4000]); print()print("=" * 78, "\nrequirement files found:")for f in glob.glob("/content/GEM-X/**/requirements*.txt", recursive=True) + \         glob.glob("/content/GEM-X/**/environment*.y*ml", recursive=True) + \         glob.glob("/content/GEM-X/pyproject.toml"):    print("  ", f)print("\ndemo scripts found:")for f in glob.glob("/content/GEM-X/scripts/demo/*.py"):    print("  ", f)

## 4. Install GEM-X dependenciesAdjust from what cell 3 printed. Colab already ships torch+CUDA, so avoidreinstalling torch unless GEM-X demands a specific version — that usually breaksthe runtime and costs you a restart.

In [ ]:
import osos.chdir("/content/GEM-X")req = Nonefor cand in ["requirements.txt", "requirements/demo.txt", "requirements/base.txt"]:    if os.path.exists(cand):        req = cand; breakif req:    print("installing from", req)    !grep -viE "^torch($|[=<>])|^torchvision|^torchaudio" $req > /tmp/req_notorch.txt    !pip install -q -r /tmp/req_notorch.txtelif os.path.exists("pyproject.toml"):    !pip install -q -e .else:    print("No requirements file found — install manually from cell 3 output.")# Common deps for monocular pose + retarget stacks:!pip install -q opencv-python-headless imageio imageio-ffmpeg scipy pandas pyyaml tqdmprint("\ndone — if you saw dependency conflicts, read them before continuing")

## 5. Model checkpointsGEM-X needs pretrained weights. I don't know its download procedure — check theREADME output from cell 3. Many NVlabs repos ship a `scripts/download_*.sh` orexpect a HuggingFace/NGC pull. Run whatever it specifies, then set `CKPT` below.

In [ ]:
import glob, osfor f in glob.glob("/content/GEM-X/**/download*.sh", recursive=True) + \         glob.glob("/content/GEM-X/**/*.sh", recursive=True)[:10]:    print("script:", f)# after downloading, point CKPT at the weights (or leave None to use GEM-X's default)CKPT = Nonefound = glob.glob("/content/GEM-X/**/*.pth", recursive=True) + \        glob.glob("/content/GEM-X/**/*.pt", recursive=True) + \        glob.glob("/content/GEM-X/**/*.onnx", recursive=True)print("\ncheckpoints present:", found if found else "NONE — download them first")

## 6. Get your videos inPick **one** option. Name each file `<motion_id>.mp4` — that stem flows through thewhole pipeline and is what pairs human↔robot in the review page.

In [ ]:
import osVID_DIR = "/content/videos"os.makedirs(VID_DIR, exist_ok=True)# ---- OPTION A: upload from your machine -----------------------------------# from google.colab import files# up = files.upload()# for name in up:#     os.rename(name, os.path.join(VID_DIR, name))# ---- OPTION B: Google Drive ------------------------------------------------# from google.colab import drive# drive.mount('/content/drive')# !cp /content/drive/MyDrive/KalariVideos/*.mp4 $VID_DIR/# ---- OPTION C: HuggingFace dataset ----------------------------------------# !pip -q install huggingface_hub# from huggingface_hub import snapshot_download# snapshot_download(repo_id="USER/DATASET", repo_type="dataset",#                   allow_patterns=["*.mp4","**/*.mp4"],#                   local_dir=VID_DIR, token="hf_xxx")vids = sorted(f for f in os.listdir(VID_DIR) if f.lower().endswith((".mp4",".mov",".webm",".mkv",".avi")))print(f"{len(vids)} videos in {VID_DIR}")for v in vids[:80]: print("  ", v)

## 7. Screen the clips before spending GPU timeGEM-X does monocular pose estimation — it fails on multi-person, cropped-feet, orvery short clips. Check the flags here first; fixing a clip is cheaper thanretargeting it twice.

In [ ]:
import cv2, osrows = []for v in vids:    p = os.path.join(VID_DIR, v)    cap = cv2.VideoCapture(p)    fps = cap.get(cv2.CAP_PROP_FPS) or 0    n   = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))    cap.release()    dur = n / fps if fps else 0    flags = []    if h < 720:  flags.append("low-res")    if fps < 24: flags.append("low-fps")    if dur > 20: flags.append("long (segment it)")    if dur < 2:  flags.append("very short")    rows.append((v, f"{w}x{h}", f"{fps:.0f}", f"{dur:.1f}s", ", ".join(flags) or "ok"))print(f"{'file':44s} {'res':>10s} {'fps':>5s} {'dur':>7s}  flags")for r in rows: print(f"{r[0][:44]:44s} {r[1]:>10s} {r[2]:>5s} {r[3]:>7s}  {r[4]}")

## 8. Batch retarget — the main runMirrors `scripts/run_pipeline.py` exactly:```python GEM-X/scripts/demo/demo_soma.py --video <v> --output_root cloud_outputs        --video-name <id> --retarget [--static_cam] [--ckpt ...]```Failures are caught per video so one bad clip doesn't kill the batch.Set `STATIC_CAM=True` if the camera doesn't move — it markedly improves the root track.

In [ ]:
import os, subprocess, sys, time, jsonos.chdir("/content/GEM-X")OUT_ROOT   = "/content/cloud_outputs"STATIC_CAM = True     # True when the camera is fixedUSE_ONNX   = False    # keep False on GPUos.makedirs(OUT_ROOT, exist_ok=True)demo = "scripts/demo/demo_soma_onnx.py" if USE_ONNX else "scripts/demo/demo_soma.py"assert os.path.exists(demo), f"demo script missing: {demo} (check cell 3 output)"results = []t0 = time.time()for i, v in enumerate(vids, 1):    mid  = os.path.splitext(v)[0]    cmd  = [sys.executable, demo,            "--video", os.path.join(VID_DIR, v),            "--output_root", OUT_ROOT,            "--video-name", mid,            "--retarget"]    if STATIC_CAM: cmd.append("--static_cam")    if CKPT:       cmd += ["--ckpt", CKPT]    print(f"\n[{i}/{len(vids)}] {mid}")    p = subprocess.run(cmd, capture_output=True, text=True)    ok = p.returncode == 0    if not ok:        print("  FAILED rc=", p.returncode)        print("  stderr tail:", p.stderr[-1200:])    results.append({"motion_id": mid, "ok": ok,                    "stderr_tail": "" if ok else p.stderr[-2000:]})print(f"\n=== {sum(r['ok'] for r in results)}/{len(results)} succeeded "      f"in {(time.time()-t0)/60:.1f} min ===")json.dump(results, open(f"{OUT_ROOT}/retarget_log.json","w"), indent=2)

## 9. Which CSVs actually got produced?

In [ ]:
import glob, os, pandas as pdcsvs = sorted(glob.glob(f"{OUT_ROOT}/**/*_retarget_g1*.csv", recursive=True))print(f"{len(csvs)} retarget CSVs found\n")for c in csvs:    try:        df = pd.read_csv(c, nrows=5)        n  = sum(1 for _ in open(c)) - 1        print(f"  {os.path.basename(c):52s} {n:5d} rows  {len(df.columns):3d} cols")    except Exception as e:        print(f"  {os.path.basename(c):52s} UNREADABLE: {e}")missing = [r["motion_id"] for r in results if not r["ok"]]if missing: print("\nfailed motions:", missing)

## 10. CSV → NPZUses **your** `annotate_motion_library.py` unchanged. Needs the G1 URDF + meshespresent at `/content/kalariSena/assets/unitree_g1/`.Check `--angles-deg` vs `--angles-rad` and `--fps` against your CSVs — thepreprocessing report used 30 fps and cm→m / deg→rad conversion.

In [ ]:
import osos.chdir("/content/kalariSena")!pip -q install pin shapely 2>/dev/null!python scripts/annotate_motion_library.py \    --csv-root /content/cloud_outputs \    --output-dir data/motions_retargeted \    --splits-dir data/splits \    --urdf assets/unitree_g1/g1.urdf \    --fps 30 --pos-scale 1.0 \    --root-euler-order xyz --quat-order xyzw \    --angles-deg!ls -la data/motions_retargeted | head -80

## 11. Package and downloadGrab the NPZs **and** the raw CSVs (so you never have to re-run the GPU step).

In [ ]:
import shutil, osos.chdir("/content")os.makedirs("/content/deliver", exist_ok=True)!cp -r /content/kalariSena/data/motions_retargeted /content/deliver/ 2>/dev/null!mkdir -p /content/deliver/retarget_csv!find /content/cloud_outputs -name "*_retarget_g1*.csv" -exec cp {} /content/deliver/retarget_csv/ \;!cp /content/cloud_outputs/retarget_log.json /content/deliver/ 2>/dev/nullshutil.make_archive("/content/kalari_retargets", "zip", "/content/deliver")print("\ncontents:")!unzip -l /content/kalari_retargets.zip | tail -20from google.colab import filesfiles.download("/content/kalari_retargets.zip")

## 12. Back on your Mac```bashcd /Users/prajaksen/Desktop/kalarisena/kalariSenaunzip ~/Downloads/kalari_retargets.zip -d .# robot videos from the NPZs.venv/bin/python scripts/render_g1_motion.py \    --npz-dir data/motions_retargeted --out results_review# side-by-side review page (put the human videos in data/kalari_videos/,# named identically to the motion ids).venv/bin/python scripts/build_review_page.py \    --human-dir data/kalari_videos --out results_review/review.htmlopen results_review/review.html```Both scripts are already built and tested. Once the NPZs land, the deliverable isthese two commands.---### If cells 3-5 fight youThat's the expected failure point. In order of likelihood:1. **torch version conflict** — Colab's torch is usually fine; don't let GEM-X   downgrade it unless it genuinely refuses to run.2. **Missing checkpoints** — GEM-X won't run without weights. Its README is   authoritative, not me.3. **A submodule didn't clone** — `cd /content/GEM-X && git submodule update --init --recursive`4. **CUDA/driver mismatch** — restart the runtime and rerun from cell 1.Paste the actual error back to me and I'll work through it with you.